In [63]:
%pip install sentence-transformers

  Obtaining dependency information for sentence-transformers from https://files.pythonhosted.org/packages/76/c1/dc1582b79e9a2eb0cddf9559cd9bcdff084f541d6fe881fdd9d98630dba7/sentence_transformers-5.6.0-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
    --------------------------------------- 10.2/596.4 kB ? eta -:--:--
    --------------------------------------- 10.2/596.4 kB ? eta -:--:--
   -- ------------------------------------ 30.7/596.4 kB 262.6 kB/s eta 0:00:03
   ---- ---------------------------------- 61.4/596.4 kB 297.7 kB/s eta 0:00:02
   ------ -------------------------------- 92.2/596.4 kB 403.5 kB/s eta 0:00:02
   --------- ---------------------------- 143.4/596.4 kB 502.3 kB/s eta 0:00:01
   --------------- ---------------------- 235.5/596.4 kB 722.1 kB/s eta 0:00:01
   ------------------- ------------------ 307.2/596.4 kB 827.2 kB/s eta 0:00:01
   ----------------------------------- ---- 522.2/596.4 kB 1.3 MB/s eta 0:00:0


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
from datasets import load_dataset


dataset = load_dataset('csv', data_files={
    'train': r'D:\Placement Prep\Projects\MCQ Solver\MCQ-Solver\dataset\train.csv',
    'test': r'D:\Placement Prep\Projects\MCQ Solver\MCQ-Solver\dataset\test.csv'
})

In [18]:
dataset = dataset.map(
    lambda x: {
        'combined_text' : x['prompt']+' '+x['A']
    }
)

Map: 100%|██████████| 500/500 [00:00<00:00, 7692.61 examples/s]


In [19]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'combined_text'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'combined_text'],
        num_rows: 500
    })
})

In [21]:
print('>> ---',len(dataset['train'][51]['combined_text']))

>> --- 614


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer)

d:\Placement Prep\Projects\MCQ Solver\MCQ-Solver\venv_mcq\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Swastik\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


In [27]:
print('>> ---', len(tokenizer.vocab))

>> --- 30522


In [31]:
print('>> ---', tokenizer.vocab['[SEP]'])


>> --- 102


In [34]:
dataset['train']['prompt']

Column(["Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.", 'What is accelerator-based light-ion fusion?', 'Determine the correct option: What is the term used in astrophysics to describe light-matter interactions resulting in energy shifts in the radiation field? among the listed options.', "Select the most accurate option: What is Martin Heidegger's view on the relationship between time and human existence? carefully.", "Identify the correct statement: What is the concept of simultaneity in Einstein's book, Relativity? carefully.", ...])

In [35]:
tokenized_prompt = tokenizer(
    list(dataset['train']['prompt']),
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

In [40]:
tokenized_prompt['input_ids'].shape

torch.Size([2000, 128])

In [ ]:
from transformers import AutoConfig

model = AutoConfig.from_pretrained("bert-base-uncased")

print("Hidden size:", model.hidden_size)
print("Attention heads:", model.num_attention_heads)

head_dim = model.hidden_size // model.num_attention_heads
print(">> ---", head_dim)

Hidden size: 768
Attention heads: 12
>> --- 64


In [46]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

text = dataset["train"][0]["prompt"]

inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs)

print('>> ---' ,outputs.last_hidden_state.shape)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4217.72it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


>> --- torch.Size([1, 31, 768])


In [50]:
cls_embedding = outputs.last_hidden_state[0, 0]
answer = cls_embedding[:5].sum().item()
print(round(answer, 4))

-1.2001


In [53]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

text = "Light-ion fusion is a technique."

inputs = tokenizer(text, return_tensors="pt")

print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

with torch.no_grad():
    outputs = model(**inputs)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3924.63it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']


In [54]:
outputs

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.3931, -0.4838, -0.1953,  ..., -0.1816, -0.1346,  0.4783],
         [-0.4813,  0.4336, -0.0777,  ..., -0.0330,  0.4466,  0.1741],
         [-0.2567,  0.6709,  0.2681,  ..., -0.2975,  0.1877,  0.1821],
         ...,
         [ 0.1136, -0.2688, -0.1499,  ..., -0.4704,  0.0810, -0.2101],
         [ 0.6836,  0.0896, -0.3702,  ...,  0.3699, -0.4204, -0.5346],
         [ 0.0727,  0.2159,  0.1718,  ...,  0.3537, -0.6438, -0.6417]]]), pooler_output=tensor([[-8.7490e-01, -2.8668e-01, -6.4522e-01,  5.6354e-01,  5.7943e-01,
         -1.3712e-01,  2.7429e-01,  1.1073e-01, -3.1853e-01, -9.9990e-01,
         -3.7329e-01,  7.1099e-01,  9.6939e-01, -1.9600e-01,  6.3388e-01,
         -6.0485e-01, -1.2704e-01, -4.3531e-01,  1.1380e-01,  2.7063e-01,
          2.9333e-01,  9.9985e-01,  2.9169e-01,  2.6529e-01,  2.2643e-01,
          8.0794e-01, -5.6697e-01,  8.0702e-01,  8.8783e-01,  6.0111e-01,
         -3.2244e-01,  1.0895e-01, -

In [58]:
attn = outputs.attentions[-1]
attn

tensor([[[[1.1480e-01, 4.0943e-02, 6.9793e-02,  ..., 1.0908e-01,
           3.0287e-01, 1.5664e-01],
          [5.4153e-03, 2.0259e-02, 2.6060e-02,  ..., 6.2131e-03,
           7.0357e-01, 1.8678e-01],
          [2.9972e-03, 1.6922e-02, 2.1000e-02,  ..., 5.9603e-03,
           7.0459e-01, 1.9628e-01],
          ...,
          [1.3843e-02, 2.3279e-02, 4.9761e-02,  ..., 3.2231e-02,
           5.0153e-01, 1.8147e-01],
          [4.1948e-03, 3.8296e-03, 2.2662e-03,  ..., 2.7414e-03,
           7.6639e-01, 2.0942e-01],
          [3.3024e-03, 3.4282e-03, 2.1241e-03,  ..., 2.2292e-03,
           7.8723e-01, 1.9201e-01]],

         [[2.2853e-01, 8.9293e-02, 2.3003e-02,  ..., 7.0676e-02,
           1.6908e-01, 1.0769e-01],
          [4.6574e-03, 4.2919e-03, 5.3778e-03,  ..., 1.0117e-02,
           6.6457e-01, 2.7828e-01],
          [2.8651e-03, 1.3187e-02, 5.2505e-03,  ..., 2.4537e-02,
           6.0609e-01, 2.7164e-01],
          ...,
          [5.3179e-02, 1.2432e-02, 7.3799e-03,  ..., 4.0728

In [56]:
tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

for i, token in enumerate(tokens):
    print(i, token)

0 [CLS]
1 light
2 -
3 ion
4 fusion
5 is
6 a
7 technique
8 .
9 [SEP]


In [60]:
head0 = attn[0, 0]
head0

tensor([[0.1148, 0.0409, 0.0698, 0.0579, 0.1025, 0.0245, 0.0211, 0.1091, 0.3029,
         0.1566],
        [0.0054, 0.0203, 0.0261, 0.0314, 0.0172, 0.0015, 0.0016, 0.0062, 0.7036,
         0.1868],
        [0.0030, 0.0169, 0.0210, 0.0260, 0.0210, 0.0026, 0.0026, 0.0060, 0.7046,
         0.1963],
        [0.0038, 0.0074, 0.0094, 0.0102, 0.0080, 0.0014, 0.0011, 0.0021, 0.7684,
         0.1882],
        [0.0076, 0.0101, 0.0214, 0.0278, 0.0436, 0.0030, 0.0032, 0.0079, 0.6978,
         0.1775],
        [0.0208, 0.0280, 0.0470, 0.0395, 0.0672, 0.0144, 0.0119, 0.0348, 0.5359,
         0.2005],
        [0.0118, 0.0293, 0.0409, 0.0384, 0.0636, 0.0111, 0.0089, 0.0310, 0.5714,
         0.1937],
        [0.0138, 0.0233, 0.0498, 0.0391, 0.1433, 0.0079, 0.0076, 0.0322, 0.5015,
         0.1815],
        [0.0042, 0.0038, 0.0023, 0.0025, 0.0041, 0.0028, 0.0018, 0.0027, 0.7664,
         0.2094],
        [0.0033, 0.0034, 0.0021, 0.0027, 0.0040, 0.0018, 0.0012, 0.0022, 0.7872,
         0.1920]])

In [62]:
weight = head0[0, 4].item()

print(weight)
print(round(weight, 4))

0.10247279703617096
0.1025


In [66]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")

prompt = dataset["train"][0]["prompt"]
option_b = dataset["train"][0]["B"]

prompt_emb = model.encode(prompt, convert_to_tensor=True)
option_b_emb = model.encode(option_b, convert_to_tensor=True)

similarity = util.cos_sim(prompt_emb, option_b_emb)

# print(similarity)
print('>> ---',round(similarity.item(), 4))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4086.82it/s]


>> --- 0.7658


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5088.56it/s]


tensor([[0.7308]])
0.7308


: 

In [ ]:
texts = [prompt, A, B, C, D, E]

tfidf = TfidfVectorizer()

X = tfidf.fit_transform(texts)

prompt_vec = X[0]
option_vecs = X[1:]

scores = cosine_similarity(prompt_vec, option_vecs)[0]

ranked = np.argsort(scores)[::-1]

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [4]:
import pandas as pd
train_df = pd.read_csv(r'D:\Placement Prep\Projects\MCQ Solver\MCQ-Solver\dataset\train.csv')
test_df = pd.read_csv(r'D:\Placement Prep\Projects\MCQ Solver\MCQ-Solver\dataset\test.csv')

In [5]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm

tqdm.pandas()

# =====================================================
# LOAD DATA
# =====================================================
# train_df = pd.read_csv("train.csv")

options = ['A', 'B', 'C', 'D', 'E']
labels = ['A', 'B', 'C', 'D', 'E']

# =====================================================
# TF-IDF SETUP (GLOBAL VOCAB)
# =====================================================

corpus = (
    train_df['prompt'] + " " +
    train_df['A'] + " " +
    train_df['B'] + " " +
    train_df['C'] + " " +
    train_df['D'] + " " +
    train_df['E']
)

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(corpus)

print(">> TF-IDF vocab size:", len(vectorizer.get_feature_names_out()))

# =====================================================
# TF-IDF PIPELINE (TOP-3)
# =====================================================

def tfidf_top3(row):
    prompt_vec = vectorizer.transform([row['prompt']])

    scores = []

    for opt in options:
        opt_vec = vectorizer.transform([row[opt]])
        score = cosine_similarity(prompt_vec, opt_vec)[0][0]
        scores.append(score)

    ranked_idx = np.argsort(scores)[::-1][:3]
    return [labels[i] for i in ranked_idx]

train_df["tfidf_top3"] = train_df.progress_apply(tfidf_top3, axis=1)

# =====================================================
# MiniLM SETUP
# =====================================================

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# =====================================================
# MiniLM PIPELINE (TOP-3)
# =====================================================

def minilm_top3(row):
    prompt_emb = model.encode(row['prompt'], convert_to_tensor=True)

    options_text = [row[o] for o in options]
    option_embs = model.encode(options_text, convert_to_tensor=True)

    scores = util.cos_sim(prompt_emb, option_embs)[0]

    ranked_idx = scores.argsort(descending=True).cpu().numpy()[:3]

    return [labels[i] for i in ranked_idx]

train_df["minilm_top3"] = train_df.progress_apply(minilm_top3, axis=1)

# =====================================================
# MAP@3 FUNCTION
# =====================================================

def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    if actual in predicted:
        rank = predicted.index(actual) + 1
        return 1.0 / rank
    return 0.0

# =====================================================
# EVALUATION
# =====================================================

tfidf_map = train_df.apply(lambda r: apk(r['answer'], r['tfidf_top3']), axis=1).mean()
minilm_map = train_df.apply(lambda r: apk(r['answer'], r['minilm_top3']), axis=1).mean()

print("\n================ RESULTS ================")
print("TF-IDF MAP@3  :", tfidf_map)
print("MiniLM MAP@3  :", minilm_map)

# =====================================================
# IMPROVEMENT COUNT
# =====================================================

improved = 0

for _, row in train_df.iterrows():
    if (row['answer'] not in row['tfidf_top3']) and (row['answer'] in row['minilm_top3']):
        improved += 1

print("\nMiniLM rescued cases:", improved)

# =====================================================
# SAMPLE OUTPUT
# =====================================================

print("\nSample predictions:")
print(train_df[['prompt', 'answer', 'tfidf_top3', 'minilm_top3']].head(3))

>> TF-IDF vocab size: 2762


100%|██████████| 2000/2000 [02:32<00:00, 13.09it/s]



================ RESULTS ================
TF-IDF MAP@3  : 0.25525
MiniLM MAP@3  : 0.4230833333333333

MiniLM rescued cases: 613

Sample predictions:
                                              prompt answer tfidf_top3  \
0  Pick the best possible answer: What is Martin ...      B  [C, D, B]   
1        What is accelerator-based light-ion fusion?      A  [C, B, A]   
2  Determine the correct option: What is the term...      C  [E, D, C]   

  minilm_top3  
0   [C, D, B]  
1   [B, C, D]  
2   [B, A, D]  
